# Gnani STT - is auto language detection correct? (10 languages)

**What the Gnani docs say (checked Oct 2026):**
- 10 languages: `bn-IN en-IN gu-IN hi-IN kn-IN ml-IN mr-IN pa-IN ta-IN te-IN`.
- REST (`POST https://api.vachana.ai/stt/v3`, header `X-API-Key-ID`, fields `audio_file`, `language_code`) and WebSocket (`wss://api.vachana.ai/stt/v3/stream`, headers `x-api-key-id`, `lang_code`).
- **Neither response contains a detected-language field** (REST returns `success, request_id, timestamp, transcript`; WS `transcript` events have `text, audio_duration_ms, segment_id, segment_index, latency`). The REST docs list no auto value; the Python SDK (v0.2.0 page) documents `AUTO_DETECT` for the streaming client as *experimental*.

**So "was the language detected correctly?" cannot be read from the API.** This notebook uses two independent kinds of evidence:
1. **Output-script inference** - 8 of the 10 languages have a unique script (Tamil, Bengali, Gujarati, Kannada, Malayalam, Telugu, Gurmukhi, Latin=English). Hindi and Marathi share Devanagari, so those two are separated with a text classifier (lower confidence, reported separately). The same inference is run on the known-language outputs (Test B) as a **calibration baseline** for the method itself.
2. **Quality gap** - CER of auto (A) vs known-language (B) on the same audio. If auto detection works, A is about as good as B.

If you see a real language field in the probe cell (cell 5), the notebook uses it directly.

**Secrets:** `GNANI_API_KEY`, `HF_TOKEN` (Colab key icon / Kaggle Add-ons -> Secrets). Never printed.

In [ ]:
!pip -q install datasets soundfile librosa jiwer requests "websockets>=14" nest_asyncio pandas scikit-learn tabulate langid huggingface_hub gnani-vachana

In [ ]:
import os, io, re, json, time, math, hashlib, unicodedata, random, asyncio
from pathlib import Path
from time import perf_counter
import numpy as np, pandas as pd, requests, soundfile as sf, librosa, websockets, nest_asyncio
nest_asyncio.apply()

def get_secret(name, default=None):
    v = os.environ.get(name)
    if v: return v
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception: pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception: pass
    return default

WORK = Path("/content/gnani_lid10") if Path("/content").exists() else (Path("/kaggle/working/gnani_lid10") if Path("/kaggle").exists() else Path("./gnani_lid10"))
for d in ["orig","wav16k"]: (WORK/d).mkdir(parents=True, exist_ok=True)
SEED = 42; random.seed(SEED)

GNANI_API_KEY = get_secret("GNANI_API_KEY"); HF_TOKEN = get_secret("HF_TOKEN")
assert GNANI_API_KEY, "Add GNANI_API_KEY secret"
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN, add_to_git_credential=False)

# ---------------- Gnani settings: make these match your production Node.js app ----------------
MODE     = "ws"                                       # "ws" (real-time agent) or "rest"
WS_URL   = "wss://api.vachana.ai/stt/v3/stream"       # your app may use /stt/v4/stream - copy from Node code
REST_URL = "https://api.vachana.ai/stt/v3"
AUTO_CODE_OVERRIDE = None   # the exact auto-detect value your prod app sends (REST has no documented one). None = use SDK AUTO_DETECT constant for ws
WORKERS  = 3                # parallel connections (raise cautiously; 429 = rate limit)
PACE     = 1.0              # WS frame pacing; 1.0 = real time (docs: do not burst). A 8 s clip takes ~8 s.

try:
    from gnani.stt import GnaniSTTStreamClient
    SDK_AUTO = getattr(GnaniSTTStreamClient, "AUTO_DETECT", None)
except Exception as e:
    SDK_AUTO = None
AUTO_CODE = AUTO_CODE_OVERRIDE or SDK_AUTO or "AUTO_DETECT"
print("MODE:", MODE, "| AUTO code used for Test A:", repr(AUTO_CODE),
      "(from SDK constant)" if (not AUTO_CODE_OVERRIDE and SDK_AUTO) else "(UNVERIFIED guess - set AUTO_CODE_OVERRIDE to what your prod app sends)" if not AUTO_CODE_OVERRIDE else "(your override)")

# ---------------- evaluation set ----------------
N_PER_LANG = 15
MIN_DUR, MAX_DUR = 1.0, 20.0
LANGS = {   # alias = substring(s) to find the Vaani config; fleurs = fallback config in google/fleurs
 "Bengali":  dict(code="bn-IN", alias=["bengali","bangla"], fleurs="bn_in"),
 "English":  dict(code="en-IN", alias=["english"],          fleurs="en_us"),
 "Gujarati": dict(code="gu-IN", alias=["gujarati"],         fleurs="gu_in"),
 "Hindi":    dict(code="hi-IN", alias=["hindi"],            fleurs="hi_in"),
 "Kannada":  dict(code="kn-IN", alias=["kannada"],          fleurs="kn_in"),
 "Malayalam":dict(code="ml-IN", alias=["malayalam"],        fleurs="ml_in"),
 "Marathi":  dict(code="mr-IN", alias=["marathi"],          fleurs="mr_in"),
 "Punjabi":  dict(code="pa-IN", alias=["punjabi","panjabi"],fleurs="pa_in"),
 "Tamil":    dict(code="ta-IN", alias=["tamil"],            fleurs="ta_in"),
 "Telugu":   dict(code="te-IN", alias=["telugu"],           fleurs="te_in"),
}
REPO_MAIN = "ARTPARK-IISc/Vaani-transcription-part"
MAX_CONFIGS_PER_LANG = 8
USE_FLEURS_FALLBACK = True      # clean read speech; only used to top up languages Vaani cannot fill
LOCAL_AUDIO_DIR = None          # optional: folder with <lang-code>/*.wav (+ same-name .txt reference), e.g. your own appliance-call clips
print("Work dir:", WORK)

## 3. Gnani client (WebSocket + REST, same return shape)

In [ ]:
def dget_langkeys(obj, found=None, prefix=""):
    # find any key that looks like a language field in any response / event (so we use it if Gnani ever returns one)
    found = {} if found is None else found
    if isinstance(obj, dict):
        for k, v in obj.items():
            if re.search(r"lang", k, re.I) and isinstance(v, (str, int, float)): found[prefix+k] = v
            elif isinstance(v, (dict, list)): dget_langkeys(v, found, prefix+k+".")
    elif isinstance(obj, list):
        for i, v in enumerate(obj): dget_langkeys(v, found, prefix)
    return found

async def ws_transcribe(wav_path, lang_code, pace=PACE, tail_s=2.0, max_extra_wait=12.0):
    y, sr = sf.read(wav_path, dtype="int16"); assert sr == 16000
    pcm = y.tobytes(); n_real = len(pcm)
    pcm += b"\x00" * int(tail_s * 16000 * 2)               # trailing silence so VAD closes the last segment
    pcm += b"\x00" * ((-len(pcm)) % 1024)
    headers = {"x-api-key-id": GNANI_API_KEY, "lang_code": lang_code, "x-sample-rate": "16000"}
    out = dict(status="error", transcript="", events=[], connect_s=None, finalize_latency_s=None, total_s=None, error=None)
    t0 = perf_counter()
    try:
        async with websockets.connect(WS_URL, additional_headers=headers, max_size=None, open_timeout=20) as ws:
            out["connect_s"] = perf_counter() - t0
            st = dict(real_end=None, sent_t=None, n_proc=0, n_tr=0, last_tr_t=None)
            async def sender():
                for i in range(0, len(pcm), 1024):
                    await ws.send(pcm[i:i+1024])
                    if st["real_end"] is None and i + 1024 >= n_real: st["real_end"] = perf_counter()
                    await asyncio.sleep(0.032 * pace)
                st["sent_t"] = perf_counter()
            async def receiver():
                while True:
                    try: m = await asyncio.wait_for(ws.recv(), 1.0)
                    except asyncio.TimeoutError:
                        if st["sent_t"] is not None and (st["n_tr"] >= st["n_proc"] or perf_counter() - st["sent_t"] > max_extra_wait): break
                        continue
                    except websockets.ConnectionClosed: break
                    if isinstance(m, bytes): continue
                    try: ev = json.loads(m)
                    except Exception: ev = {"type": "unparsed", "raw": m[:200]}
                    ev["_recv_t"] = round(perf_counter() - t0, 3); out["events"].append(ev)
                    if ev.get("type") == "processing": st["n_proc"] += 1
                    elif ev.get("type") == "transcript": st["n_tr"] += 1; st["last_tr_t"] = perf_counter()
                    elif ev.get("type") == "error": out["error"] = str(ev.get("message"))[:200]
            await asyncio.gather(sender(), receiver())
            segs = [e.get("text", "") for e in out["events"] if e.get("type") == "transcript"]
            out["transcript"] = " ".join(s for s in segs if s).strip()
            if st["last_tr_t"] and st["real_end"]: out["finalize_latency_s"] = st["last_tr_t"] - st["real_end"]
            out["status"] = "ok" if not out["error"] else "error"
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {str(e)[:200]}"
    out["total_s"] = perf_counter() - t0
    return out

def rest_transcribe(wav_path, lang_code):
    out = dict(status="error", transcript="", events=[], connect_s=None, finalize_latency_s=None, total_s=None, error=None)
    t0 = perf_counter()
    try:
        with open(wav_path, "rb") as f:
            r = requests.post(REST_URL, headers={"X-API-Key-ID": GNANI_API_KEY},
                              files={"audio_file": (Path(wav_path).name, f, "audio/wav")},
                              data={"language_code": lang_code, "format": "verbatim"}, timeout=120)
        try: body = r.json()
        except Exception: body = {"text": r.text[:300]}
        out["events"] = [body]
        if r.status_code == 200 and isinstance(body, dict) and body.get("success", True):
            out.update(status="ok", transcript=(body.get("transcript") or "").strip())
        else:
            out["error"] = f"HTTP {r.status_code}: {str(body)[:200]}"
            out["http"] = r.status_code
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {str(e)[:200]}"
    out["total_s"] = perf_counter() - t0; out["finalize_latency_s"] = out["total_s"]
    return out

async def transcribe(wav_path, lang_code):
    for attempt in (1, 2, 3):
        r = await ws_transcribe(wav_path, lang_code) if MODE == "ws" else await asyncio.get_event_loop().run_in_executor(None, rest_transcribe, wav_path, lang_code)
        r["attempts"] = attempt
        if r["status"] == "ok" or (r.get("error") and ("401" in r["error"] or "403" in r["error"] or "400" in r["error"])): return r
        await asyncio.sleep(2 * attempt)
    return r

## 4. Text normalisation, script-based language inference, scoring helpers

In [ ]:
import jiwer, unicodedata, langid
langid.set_languages(["hi","mr"])
TAG_RE = re.compile(r"<[^>]*>|\[[^\]]*\]")

def normalize(t):
    t = unicodedata.normalize("NFC", t or ""); t = TAG_RE.sub(" ", t)
    t = "".join(" " if unicodedata.category(c)[0] in "PS" else c for c in t)
    t = t.replace("\u200c", "").replace("\u200d", "")
    return re.sub(r"\s+", " ", t).strip().lower()

def cer_counts(ref, hyp):
    ref, hyp = normalize(ref), normalize(hyp)
    n = len(ref)
    if n == 0: return (0, 0)
    if not hyp: return (n, n)
    o = jiwer.process_characters(ref, hyp); return (o.substitutions + o.deletions + o.insertions, n)

SCRIPTS = [("bn-IN",0x0980,0x09FF),("pa-IN",0x0A00,0x0A7F),("gu-IN",0x0A80,0x0AFF),("ta-IN",0x0B80,0x0BFF),
           ("te-IN",0x0C00,0x0C7F),("kn-IN",0x0C80,0x0CFF),("ml-IN",0x0D00,0x0D7F),("DEVA",0x0900,0x097F)]
def infer_lang(text):
    # returns (language code inferred from OUTPUT script, dominant-script share, method)
    cnt = {}
    for ch in unicodedata.normalize("NFC", text or ""):
        if not ch.isalpha(): continue
        cp = ord(ch); key = None
        for name, lo, hi in SCRIPTS:
            if lo <= cp <= hi: key = name; break
        if key is None and ("LATIN" in unicodedata.name(ch, "")): key = "en-IN"
        if key: cnt[key] = cnt.get(key, 0) + 1
    tot = sum(cnt.values())
    if tot == 0: return (None, 0.0, "no_letters")
    dom, c = max(cnt.items(), key=lambda kv: kv[1]); share = c / tot
    if dom == "DEVA":
        lab, _ = langid.classify(" ".join(w for w in text.split() if any(0x0900 <= ord(ch) <= 0x097F for ch in w)))
        return ("hi-IN" if lab == "hi" else "mr-IN", share, "devanagari+langid")
    return (dom, share, "script")

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z*z/n; c = (p + z*z/(2*n)) / d; h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (max(0, c-h), min(1, c+h))

# quick self-test of the inference
for s in ["நான் சந்தைக்கு செல்கிறேன்","আমি ভাত খাই","मैं बाज़ार जा रहा हूँ","मी बाजारात जातोय","I am going to the market","ਮੈਂ ਬਾਜ਼ਾਰ ਜਾ ਰਿਹਾ ਹਾਂ"]:
    print(infer_lang(s), s)

## 5. Probe the API (one clip): does the response contain a detected-language field?

In [ ]:
# Build any one short wav for the probe (a 2 s tone is enough to inspect response structure; use a real clip after step 6 for transcript content)
import soundfile as sf
probe = WORK / "probe.wav"
if not probe.exists():
    t = np.linspace(0, 2, 32000, endpoint=False); sf.write(probe, (0.1*np.sin(2*np.pi*220*t)).astype("float32"), 16000, subtype="PCM_16")
r_auto  = asyncio.get_event_loop().run_until_complete(transcribe(str(probe), AUTO_CODE))
print("AUTO  ->", r_auto["status"], "|", r_auto["error"], "| events:", [e.get("type") for e in r_auto["events"]])
print("auto-mode events (raw):", json.dumps(r_auto["events"], ensure_ascii=False)[:800])
if r_auto["status"] != "ok" and "400" in (r_auto["error"] or "") or (r_auto["error"] and "AUTO" in r_auto["error"].upper()):
    print("\n!! Auto code rejected. Set AUTO_CODE_OVERRIDE to the exact value your production Node app sends, then re-run from cell 2.")
langkeys = dget_langkeys(r_auto["events"]); print("Language-like keys in response:", langkeys or "NONE (language must be inferred from output script)")

## 6. Build the evaluation set (streams a bounded number of clips per language)

In [ ]:
from datasets import load_dataset, get_dataset_config_names, get_dataset_split_names, Audio
NOISE_RE = re.compile(r"noise|unclear|inaudible|non[- ]?speech|music|laugh|cough|silence|background", re.I)
TEXT_CANDS = ["transcription","transcript","text","sentence","normalized_text","verbatim"]
SPK_CANDS = ["speaker_id","speakerid","speaker","user_id"]

def pick(cols, cands):
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c in low: return low[c]

def decode_bytes(b):
    try:
        y, sr = sf.read(io.BytesIO(b), dtype="float32", always_2d=False)
        if y.ndim > 1: y = y.mean(axis=1)
    except Exception:
        y, sr = librosa.load(io.BytesIO(b), sr=None, mono=True)
    return y.astype("float32"), int(sr)

def add_clip(rows, excl, seen, b, ref, lang, source, cfg, spk=None):
    h = hashlib.md5(b).hexdigest()
    if h in seen: excl.append((source, cfg, h[:8], "duplicate")); return False
    try: y, sr = decode_bytes(b)
    except Exception as e: excl.append((source, cfg, h[:8], "decode_failed")); return False
    dur = len(y) / sr
    if not (MIN_DUR <= dur <= MAX_DUR): excl.append((source, cfg, h[:8], f"duration {dur:.1f}s")); return False
    if not np.isfinite(y).all() or np.abs(y).max() < 1e-4: excl.append((source, cfg, h[:8], "silent")); return False
    sid = f"{LANGS[lang]['code']}-{source}-{h[:10]}"
    y16 = librosa.resample(y, orig_sr=sr, target_sr=16000) if sr != 16000 else y
    sf.write(WORK/"wav16k"/f"{sid}.wav", y16, 16000, subtype="PCM_16")     # ONE processed copy used by Test A and Test B
    seen.add(h)
    rows.append(dict(sample_id=sid, language=lang, true_code=LANGS[lang]["code"], source=source, config=cfg,
                     wav_path=str(WORK/"wav16k"/f"{sid}.wav"), duration_s=round(dur, 2), orig_sr=sr, reference=ref or "",
                     noisy_marked=bool(any(NOISE_RE.search(t) for t in TAG_RE.findall(ref or ""))), speaker_id=spk))
    return True

def stream_clips(repo, cfg, want, lang, source, rows, excl, seen, extra=None, split=None):
    got = 0
    try:
        if split is None:
            try: sp = get_dataset_split_names(repo, cfg, token=HF_TOKEN); split = "train" if "train" in sp else sp[0]
            except Exception: split = "test"
        ds = load_dataset(repo, cfg, split=split, streaming=True, token=HF_TOKEN, **(extra or {}))
        try: ds = ds.cast_column("audio", Audio(decode=False))
        except Exception: pass
        ds = ds.shuffle(seed=SEED, buffer_size=300)
    except Exception as e:
        print(f"   ! {repo}/{cfg}: {type(e).__name__}: {str(e)[:120]}"); return 0
    for k, ex in enumerate(ds):
        if got >= want or k > want * 12: break
        a = ex.get("audio")
        if not isinstance(a, dict): continue
        b = a.get("bytes") or (open(a["path"], "rb").read() if a.get("path") and os.path.exists(a["path"]) else None)
        if not b: continue
        tc = pick(list(ex.keys()), TEXT_CANDS); ref = ex.get(tc) if tc and isinstance(ex.get(tc), str) else ""
        sc = pick(list(ex.keys()), SPK_CANDS)
        got += add_clip(rows, excl, seen, b, ref, lang, source, cfg, str(ex[sc]) if sc else None)
    return got

MANIFEST = WORK / "manifest.csv"; REBUILD = False
if MANIFEST.exists() and not REBUILD:
    manifest = pd.read_csv(MANIFEST).fillna({"reference": ""}); print("loaded manifest", len(manifest))
else:
    rows, excl, seen = [], [], set()
    configs = get_dataset_config_names(REPO_MAIN, token=HF_TOKEN) if HF_TOKEN else []
    print(len(configs), "Vaani configs, e.g.", configs[:6])
    for lang, d in LANGS.items():
        have = 0
        m = [c for c in configs if any(a in c.lower() for a in d["alias"])]
        random.Random(SEED).shuffle(m); m = m[:MAX_CONFIGS_PER_LANG]
        print(f"{lang}: {len(m)} Vaani configs")
        for c in m:
            if have >= N_PER_LANG: break
            have += stream_clips(REPO_MAIN, c, math.ceil(N_PER_LANG / max(len(m), 1)), lang, "vaani", rows, excl, seen)
        if have < N_PER_LANG and USE_FLEURS_FALLBACK:
            have += stream_clips("google/fleurs", d["fleurs"], N_PER_LANG - have, lang, "fleurs", rows, excl, seen, extra=dict(trust_remote_code=True), split="test")
        if have < N_PER_LANG and LOCAL_AUDIO_DIR:
            for p in sorted((Path(LOCAL_AUDIO_DIR)/d["code"]).glob("*.wav"))[: N_PER_LANG - have]:
                ref = p.with_suffix(".txt").read_text(encoding="utf-8") if p.with_suffix(".txt").exists() else ""
                have += add_clip(rows, excl, seen, p.read_bytes(), ref, lang, "local", "local")
        print(f"   -> {have} clips")
    manifest = pd.DataFrame(rows); manifest.to_csv(MANIFEST, index=False)
    pd.DataFrame(excl, columns=["source","config","sample","reason"]).to_csv(WORK/"exclusions.csv", index=False)
print(manifest.groupby(["language","source"]).size().unstack(fill_value=0))

In [ ]:
# Sanity-check labels before spending API calls: references should be in the expected script
chk = manifest[manifest.reference.str.len() > 0].copy()
chk["ref_inferred"] = chk.reference.map(lambda t: infer_lang(t)[0])
chk["label_ok"] = chk.ref_inferred == chk.true_code
print("Reference-script agrees with dataset language label:")
print(chk.groupby("language").label_ok.mean().round(2).to_string())
print("Languages with < N clips:", {l: int(n) for l, n in manifest.groupby("language").size().items() if n < N_PER_LANG})
# Listen to a few:  from IPython.display import Audio, display; display(Audio(manifest.wav_path.iloc[0]))

## 7. Run Test A (auto) and Test B (known language) on the same wav files - resumable

In [ ]:
RES = WORK / "results.jsonl"; done = {}
if RES.exists():
    for l in RES.read_text().splitlines():
        d = json.loads(l); done[(d["sample_id"], d["test"])] = d

async def run_all(jobs):
    sem = asyncio.Semaphore(WORKERS); f = open(RES, "a"); n = 0
    async def one(job):
        nonlocal n
        sid, test, path, code_ = job
        async with sem:
            r = await transcribe(path, code_)
        rec = dict(sample_id=sid, test=test, requested=code_, status=r["status"], transcript=r["transcript"], error=r["error"],
                   total_s=r["total_s"], connect_s=r["connect_s"], finalize_latency_s=r["finalize_latency_s"], attempts=r["attempts"],
                   n_segments=sum(1 for e in r["events"] if e.get("type") == "transcript") if MODE == "ws" else 1,
                   lang_keys=json.dumps(dget_langkeys(r["events"])), events=json.dumps(r["events"], ensure_ascii=False)[:3000])
        done[(sid, test)] = rec; f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush(); n += 1
        if n % 10 == 0: print(f"  {n}/{len(jobs)} done")
    await asyncio.gather(*[one(j) for j in jobs]); f.close()

def pending(test, code_fn): return [(r.sample_id, test, r.wav_path, code_fn(r)) for r in manifest.itertuples() if (r.sample_id, test) not in done]

# smoke test: 3 clips first
sm = manifest.groupby("language").head(1).head(3)
jobs = [(r.sample_id, "A_auto", r.wav_path, AUTO_CODE) for r in sm.itertuples() if (r.sample_id,"A_auto") not in done]
asyncio.get_event_loop().run_until_complete(run_all(jobs))
for r in sm.itertuples(): x = done[(r.sample_id,"A_auto")]; print(r.language, "|", x["status"], "|", x["error"], "|", x["transcript"][:60])

In [ ]:
jobs = pending("A_auto", lambda r: AUTO_CODE) + pending("B_known", lambda r: r.true_code)
print("pending calls:", len(jobs), "| est. minutes at PACE", PACE, ":", round(sum(manifest.duration_s) * 2 / WORKERS / 60 * PACE, 1))
asyncio.get_event_loop().run_until_complete(run_all(jobs))

## 8. Analysis: was the language detected correctly?

In [ ]:
res = pd.DataFrame(list(done.values()))
df = manifest.merge(res, on="sample_id")
df["empty"] = (df.status == "ok") & (df.transcript.fillna("").str.strip() == "")
df["failed"] = df.status != "ok"
inf = df.transcript.map(lambda t: infer_lang(t))
df["inferred"] = inf.map(lambda x: x[0]); df["script_share"] = inf.map(lambda x: x[1]); df["infer_method"] = inf.map(lambda x: x[2])
df["inferred_ok"] = df.inferred == df.true_code
sc = df.apply(lambda r: cer_counts(r.reference, r.transcript) if r.reference else (np.nan, np.nan), axis=1, result_type="expand"); df[["cer_err","cer_n"]] = sc
df.to_csv(WORK/"per_sample_results.csv", index=False)

A = df[df.test == "A_auto"]; B = df[df.test == "B_known"]
reported = A.lang_keys.map(lambda s: bool(json.loads(s))).any()
print("API returned an explicit language field:", reported)

rows = []
for lang in LANGS:
    a = A[A.language == lang]; b = B[B.language == lang]
    ne = a[(~a.failed) & (~a.empty)]
    k = int(ne.inferred_ok.sum()); n = len(ne); lo, hi = wilson(k, n)
    ka = int(a.inferred_ok.sum())   # empty/failed count as wrong
    bne = b[(~b.failed) & (~b.empty)]
    rows.append(dict(language=lang, code=LANGS[lang]["code"], n_clips=len(a), fail_or_empty=int(a.failed.sum()+a.empty.sum()),
        A_detect_acc=round(k/n, 3) if n else np.nan, A_ci95=f"{lo:.2f}-{hi:.2f}", A_acc_incl_empty=round(ka/len(a), 3) if len(a) else np.nan,
        B_script_match_baseline=round(bne.inferred_ok.mean(), 3) if len(bne) else np.nan,
        A_CER=round(a.cer_err.sum()/a.cer_n.sum(), 3) if a.cer_n.sum() else np.nan,
        B_CER=round(b.cer_err.sum()/b.cer_n.sum(), 3) if b.cer_n.sum() else np.nan,
        A_latency_s=round(a.finalize_latency_s.mean(), 2), B_latency_s=round(b.finalize_latency_s.mean(), 2)))
summary = pd.DataFrame(rows); summary["CER_gap_A_minus_B"] = (summary.A_CER - summary.B_CER).round(3)
display(summary)
print("Overall A detection accuracy (non-empty):", round(A[(~A.failed)&(~A.empty)].inferred_ok.mean(), 3),
      "| B baseline (how often the inference method agrees even when language is FORCED):", round(B[(~B.failed)&(~B.empty)].inferred_ok.mean(), 3))

In [ ]:
# Confusion matrix: true language (rows) vs language inferred from auto-mode output (cols)
ne = A[(~A.failed) & (~A.empty)].copy(); ne["inferred"] = ne.inferred.fillna("none")
cm = pd.crosstab(ne.language, ne.inferred); display(cm)
conf = ne[~ne.inferred_ok].groupby(["language","inferred"]).size().sort_values(ascending=False).rename("count").reset_index(); print("Top confusions:"); display(conf.head(15))
print("By source (vaani = natural/noisier, fleurs = clean read speech):"); display(ne.groupby(["language","source"]).inferred_ok.agg(["mean","count"]).round(2).unstack("source"))
if ne.noisy_marked.any(): print("Noise-tagged vs untagged:"); display(ne.groupby(["noisy_marked"]).inferred_ok.agg(["mean","count"]).round(3))
# Hindi/Marathi are both Devanagari: how many errors are within that pair (low-confidence inference) vs cross-script (high-confidence)
dev = {"hi-IN","mr-IN"}; wrong = ne[~ne.inferred_ok]
print("Wrong, same-script Hindi<->Marathi (low confidence):", int(wrong.apply(lambda r: r.true_code in dev and r.inferred in dev, axis=1).sum()),
      "| wrong, cross-script (high confidence):", int(len(wrong) - wrong.apply(lambda r: r.true_code in dev and r.inferred in dev, axis=1).sum()))

In [ ]:
# Worst examples (cross-script confusions), with reference and the forced-language output side by side
bt = B.set_index("sample_id").transcript
ex = ne[~ne.inferred_ok].copy(); ex["B_transcript"] = ex.sample_id.map(bt)
display(ex[["sample_id","language","inferred","source","reference","transcript","B_transcript"]].head(15))

## 9. Write report files

In [ ]:
summary.to_csv(WORK/"summary_per_language.csv", index=False); cm.to_csv(WORK/"confusion_matrix.csv")
agg = dict(api_returns_language_field=bool(reported), mode=MODE, auto_code=AUTO_CODE, n_clips=int(len(manifest)),
           overall_A_detect_acc=float(A[(~A.failed)&(~A.empty)].inferred_ok.mean()), overall_B_baseline=float(B[(~B.failed)&(~B.empty)].inferred_ok.mean()),
           per_language=json.loads(summary.to_json(orient="records")))
(WORK/"aggregate_metrics.json").write_text(json.dumps(agg, ensure_ascii=False, indent=2))
rep = f"""# Gnani STT auto language detection - 10-language check
- Mode: {MODE}; auto code sent: `{AUTO_CODE}`; clips: {len(manifest)} ({N_PER_LANG}/language target)
- API returns an explicit detected-language field: **{reported}**. If False, correctness is INFERRED from the output script
  (Hindi vs Marathi via langid on Devanagari text - low confidence).
- Baseline column `B_script_match_baseline` shows how often the same inference agrees when the language is forced; accuracy in A
  cannot be meaningfully higher than that baseline.

## Per-language results
{summary.to_markdown(index=False)}

## Confusion (rows = true language, cols = language inferred from auto output)
{cm.to_markdown()}

## Limitations
Small per-language n (see CI). Script inference measures which language the model *wrote*, not an internal LID decision. Vaani clips are
natural/noisy; FLEURS fallback clips are clean read speech and English there is US English. Transcript conventions differ from Gnani output, so use the A-B CER *gap*, not absolute CER.
"""
(WORK/"report.md").write_text(rep); print(rep[:1500]); print("Files:", [p.name for p in WORK.glob("*") if p.is_file()])